# UART FPGA Test Interface

Interface Python para testar comunicação UART com a Tang Nano / MAX II via porta serial.

## Configuração
- **Baud Rate:** 115200 bps
- **Data Bits:** 8
- **Parity:** Odd (Ímpar)
- **Stop Bits:** 1
- **Porta:** `/dev/ttyUSB0` (Linux) ou `COM3` (Windows)

## Implementação com PySerial

In [23]:
import serial
import time

In [49]:
class UARTInterface:
    def __init__(self, port="/dev/ttyUSB0", baudrate=115200, timeout=1):
        """Cria conexão serial com PySerial"""
        try:
            self.ser = serial.Serial(
                port=port,
                baudrate=baudrate,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_NONE,      # Paridade ímpar ODD (FPGA usa)
                stopbits=serial.STOPBITS_ONE,
                timeout=timeout
            )
            print(f" Conectado em {port} @ {baudrate} bps (PySerial)")
            self.ser.reset_input_buffer()
            self.ser.reset_output_buffer()
        except Exception as e:
            print(f" Erro ao conectar: {e}")
            self.ser = None
    
    def send(self, data):
        """Envia dados"""
        if not self.ser:
            return False
        try:
            if isinstance(data, str):
                data = data.encode()
            self.ser.write(data)
            print(f" Enviado: {data} ({len(data)} bytes)")
            return True
        except Exception as e:
            print(f" Erro ao enviar: {e}")
            return False
    
    def receive(self, size=1):
        """Recebe dados"""
        if not self.ser:
            return None
        try:
            data = self.ser.read(size)
            if data:
                print(f" Recebido: {data} ({len(data)} bytes)")
            return data
        except Exception as e:
            print(f" Erro ao receber: {e}")
            return None
    
    def send_and_wait(self, data, wait_time=0.1):
        """Envia e aguarda resposta"""
        self.send(data)
        time.sleep(wait_time)
        return self.receive()
    
    def send_string(self, text):
        """Envia string"""
        return self.send(text.encode())
    
    def receive_string(self, size=1):
        """Recebe como string"""
        data = self.receive(size)
        return data.decode() if data else None
    
    def is_connected(self):
        """Verifica se está conectado"""
        return self.ser is not None and self.ser.is_open
    
    def close(self):
        """Fecha conexão"""
        if self.ser:
            self.ser.close()
            print(" Conexão fechada")

In [50]:
baudrate = 115200
port = "/dev/ttyUSB0"

In [51]:
print("TESTE COM PySerial")
try:
    # Teste Básico
    print("\n Teste Básico")
    uart = UARTInterface(port=port, baudrate=baudrate)
    
    if uart.is_connected():
        # Enviar um byte
        uart.send(b'\x41')  # ASCII 'A'
        time.sleep(0.2)
        
        # Receber
        response = uart.receive(1)
        
        uart.close()
    
    # Teste com String
    print("\n Teste com String")
    uart = UARTInterface(port=port, baudrate=115200)
    
    if uart.is_connected():
        uart.send_string("Hello FPGA!")
        time.sleep(0.5)
        response = uart.receive_string(20)
        uart.close()
    
    # Teste Múltiplos Bytes
    print("\n Teste Múltiplos Bytes")
    uart = UARTInterface(port=port, baudrate=baudrate)
    
    if uart.is_connected():
        test_data = bytes([0x01, 0x02, 0x03, 0x04, 0x05])
        uart.send(test_data)
        time.sleep(0.5)
        response = uart.receive(5)
        uart.close()
        
except Exception as e:
    print(f"Erro: {e}")

TESTE COM PySerial

 Teste Básico
 Conectado em /dev/ttyUSB0 @ 115200 bps (PySerial)
 Enviado: b'A' (1 bytes)
 Recebido: b'\xfa' (1 bytes)
 Conexão fechada

 Teste com String
 Conectado em /dev/ttyUSB0 @ 115200 bps (PySerial)
 Enviado: b'Hello FPGA!' (11 bytes)
 Recebido: b'\xfaH\xfae\xfal\xfal\xff\xfaP\xfaG\xfaA\xfa!' (17 bytes)
Erro: 'utf-8' codec can't decode byte 0xfa in position 0: invalid start byte


## Referência Rápida de Portas

- **Linux:** `/dev/ttyUSB0`, `/dev/ttyUSB1`, `/dev/ttyACM0`
- **Windows:** `COM3`, `COM4`, `COM5`
- **macOS:** `/dev/cu.usbserial-*`

### Como Encontrar a Porta?

**Linux:**
```bash
ls /dev/tty*
dmesg | grep -i usb
```

**Windows:**
```cmd
mode COM3
```

**macOS:**
```bash
ls -la /dev/cu.*
```

In [28]:
"""
Teste Interativo - Use esta célula para testes manuais
"""

# Configurações
PORT = "/dev/ttyUSB1"
BAUDRATE = 115200

# Criar instância global
uart = UARTInterface(port=PORT, baudrate=BAUDRATE)

 Conectado em /dev/ttyUSB1 @ 115200 bps (PySerial)


In [29]:
def test_loopback():
    """Testa se o FPGA está ecoando os dados (loopback)"""
    print("\n Testando Loopback...")
    test_messages = [
        b'A',
        b'Hello',
        bytes([0x01, 0x02, 0x03]),
        b'UART_TEST'
    ]
    
    for msg in test_messages:
        print(f"\nEnviando: {msg}")
        uart.send(msg)
        time.sleep(0.3)
        response = uart.receive(len(msg))
        
        if response == msg:
            print(" Loopback OK!")
        else:
            print(f" Esperado {msg}, recebido {response}")

In [30]:
def test_baud_sync():
    """Testa sincronização de baud rate"""
    print("\n Testando Sincronização...")
    
    # Enviar sequência conhecida
    test_seq = b'\x55'  # 01010101 - padrão alternado
    uart.send(test_seq)
    time.sleep(0.5)
    response = uart.receive(1)
    
    if response:
        print(f" Dados recebidos com sucesso")
    else:
        print(" Timeout - verificar baud rate ou conexão")

In [31]:
def manual_send(msg):
    """Envia mensagem manualmente"""
    if isinstance(msg, str):
        uart.send_string(msg)
    else:
        uart.send(msg)
    time.sleep(0.2)
    return uart.receive(len(msg) if isinstance(msg, bytes) else len(msg.encode()))

In [ ]:
def test_baud_rates():
    """Testa múltiplos baud rates para diagnosticar sincronização"""
    test_rates = [9600, 19200, 38400, 57600, 115200]
    test_byte = b'A'  # 0x41
    
    for rate in test_rates:
        try:
            uart_test = UARTInterface(port=port, baudrate=rate, timeout=0.5)
            if uart_test.is_connected():
                uart_test.send(test_byte)
                time.sleep(0.3)
                response = uart_test.receive(1)
                
                status = "✓ OK" if response == test_byte else "✗ FAIL"
                print(f"{rate:6d} bps: Enviado {test_byte.hex():>4s} → Recebido {response.hex() if response else '(timeout)':>4s}  {status}")
                uart_test.close()
        except Exception as e:
            print(f"{rate:6d} bps: Erro - {e}")
        time.sleep(0.1)

print("Testando múltiplos baud rates...")
test_baud_rates()


 Testando Loopback...

Enviando: b'A'
 Enviado: b'A' (1 bytes)
 Recebido: b'\xfa' (1 bytes)
 Esperado b'A', recebido b'\xfa'

Enviando: b'Hello'
 Enviado: b'Hello' (5 bytes)
 Recebido: b'A\xfaH\xfae' (5 bytes)
 Esperado b'Hello', recebido b'A\xfaH\xfae'

Enviando: b'\x01\x02\x03'
 Enviado: b'\x01\x02\x03' (3 bytes)
 Recebido: b'\xfal\xfa' (3 bytes)
 Esperado b'\x01\x02\x03', recebido b'\xfal\xfa'

Enviando: b'UART_TEST'
 Enviado: b'UART_TEST' (9 bytes)
 Recebido: b'l\xc0\xfa\x03\xfaU\xfaA\xfa' (9 bytes)
 Esperado b'UART_TEST', recebido b'l\xc0\xfa\x03\xfaU\xfaA\xfa'


## Troubleshooting

### Problema: Porta não encontrada

```python
# Listar portas disponíveis
import serial.tools.list_ports
ports = serial.tools.list_ports.comports()
for port in ports:
    print(f"{port.device}: {port.description}")
```

### Problema: Timeout ao receber

- Verificar se FPGA está programado e rodando
- Verificar baud rate (deve ser 115200)
- Verificar parity (deve ser ODD - ímpar)
- Testar com `minicom` ou `screen`

### Problema: Dados corrompidos

- Verificar cabos USB
- Aumentar `timeout`
- Verificar polaridade RX/TX
- Verificar se baud rate está sincronizado entre FPGA e PySerial

### Teste com Terminal

```bash
# Linux
minicom -D /dev/ttyUSB0 -b 115200
# ou
screen /dev/ttyUSB0 115200
```

---

## Estrutura UART do Projeto

Seu FPGA implementa:
- **TX:** Transmissão de dados (FSM com 4 estados)
- **RX:** Recepção de dados  
- **Parity:** Bit de paridade ímpar (ODD) - XOR de todos os bits
- **Frame:** START + 8 bits + PARITY + STOP